In [1]:
import pickle
import pandas as pd
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
import sqlite3
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnablePassthrough

# load chanks
LOAD_PATH = Path("../data/processed/chunks.pkl")

with open(LOAD_PATH, "rb") as f:
    chunks = pickle.load(f)

print(f"✅ Loaded {len(chunks)} chunks")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("✅ Embedding Model Loaded")

vectorstore = FAISS.load_local(
    "../data/processed/vectorstore",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("✅ FAISS Loaded")

# vector retrival(summantic ret)
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Vector Retriever Ready")

# create BM25 retrival
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

print("✅ BM25 Ready")

# hybread retrival
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        vector_retriever
    ],
    weights=[
        0.4,
        0.6
    ]
)

print("✅ Hybrid Retriever Ready")

from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
print(api_key)   # Test ke liye

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are a professional customer support assistant build by saurabh upadhyay.

Answer ONLY from the provided context.

Rules:
1. Never make up information.
2. If the answer is not available in the context, say:
   "I couldn't find the correct information. I'll connect you with a human support agent."
3. Be polite.
4. Keep answers concise.
5. Guide the customer step by step whenever appropriate.

Context:
{context}

Question:
{question}

Answer:
""")

# context formet
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain Ready")

DATABASE_PATH = r"C:\Users\R&D\Desktop\AI System\GENAI\DataLoading\orders.db"

conn = sqlite3.connect(DATABASE_PATH)

df = pd.read_sql("PRAGMA table_info(orders);", conn)

conn.close()

conn = sqlite3.connect(DATABASE_PATH)

print("===== ORDERS TABLE =====")
print(pd.read_sql("SELECT * FROM orders LIMIT 5;", conn))

print("\n===== CUSTOMERS TABLE =====")
print(pd.read_sql("SELECT * FROM customers LIMIT 5;", conn))

print("\n===== PRODUCTS TABLE =====")
print(pd.read_sql("SELECT * FROM products LIMIT 5;", conn))

conn.close()

@tool
def get_order_status(order_id: str):
    """
    Get complete order details using Order ID.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            c.name,
            p.name,
            o.quantity,
            o.order_date,
            o.delivery_date,
            o.status,
            o.payment_method,
            o.tracking_id

        FROM orders o

        JOIN customers c
            ON o.customer_id = c.customer_id

        JOIN products p
            ON o.product_id = p.product_id

        WHERE o.order_id = ?
    """, (order_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return "❌ Order not found."

    return f"""
Order ID: {order_id}

Customer: {row[0]}

Product: {row[1]}

Quantity: {row[2]}

Order Date: {row[3]}

Delivery Date: {row[4]}

Status: {row[5]}

Payment Method: {row[6]}

Tracking ID: {row[7]}
"""

@tool
def check_return(order_id:str):
    """
    Check whether order is eligible for return.
    """

    conn=sqlite3.connect(DATABASE_PATH)

    cursor=conn.cursor()

    cursor.execute("""
    SELECT
    return_window,
    return_status

    FROM orders

    WHERE order_id=?
    """,(order_id,))

    row=cursor.fetchone()

    conn.close()

    if row is None:

        return "Order not found."

    return f"""

Return Window : {row[0]}

Return Status : {row[1]}
"""
@tool
def track_delivery(order_id: str):
    """
    Get delivery status using order id.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            status,
            delivery_date,
            tracking_id

        FROM orders

        WHERE order_id=?
    """, (order_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return "Order not found."

    return f"""
Order ID : {order_id}

Status : {row[0]}

Expected Delivery : {row[1]}

Tracking ID : {row[2]}
"""

@tool
def get_product_details(order_id: str):
    """
    Get ordered product details using order id.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            p.name,
            p.category,
            p.price,
            o.quantity

        FROM orders o

        JOIN products p
        ON o.product_id=p.product_id

        WHERE o.order_id=?
    """, (order_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return "Order not found."

    total = row[2] * row[3]

    return f"""
Product : {row[0]}

Category : {row[1]}

Price : ₹{row[2]}

Quantity : {row[3]}

Total Price : ₹{total}
"""
@tool
def get_customer_details(order_id: str):
    """
    Get customer information using order id.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            c.name,
            c.email,
            c.phone,
            c.address

        FROM orders o

        JOIN customers c
        ON o.customer_id=c.customer_id

        WHERE o.order_id=?
    """, (order_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return "Order not found."

    return f"""
Customer : {row[0]}

Email : {row[1]}

Phone : {row[2]}

Address :

{row[3]}
"""
tools = [
    get_order_status,
    track_delivery,
    get_product_details,
    get_customer_details,
    
]

llm_with_tools = llm.bind_tools(tools)
   

print("Tools Connected")



c:\Users\R&D\Desktop\AI System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\R&D\AppData\Local\Temp\ipykernel_10000\1416021541.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Loaded 136 chunks


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [14]:
# Create Tool Dictionary
# Agent ko tool execute karne ke liye mapping chahiye:
tool_dict = {
    tool.name: tool 
    for tool in tools
}

In [15]:
print(tool_dict.keys())

dict_keys(['get_order_status', 'track_delivery', 'get_product_details', 'get_customer_details'])


In [16]:
# agent function
def customer_support_agent(question):

    messages = [
        SystemMessage(
            content="""
You are an Amazon style customer support assistant.

Use tools whenever required.
Never guess order information.
Always use database tools for customer data.
"""
        ),
        HumanMessage(content=question)
    ]


    response = llm_with_tools.invoke(messages)


    # If tool required
    if response.tool_calls:

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]

            tool_args = tool_call["args"]


            tool = tool_dict[tool_name]


            tool_result = tool.invoke(tool_args)


            messages.append(response)


            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )


        final_response = llm.invoke(messages)

        return final_response.content


    else:
        return response.content

In [20]:
while True:

    user=input("Customer: ")

    if user=="exit":
        break


    answer = customer_support_agent(user)


    print("\nBot:",answer)


Bot: I’m sorry to hear that you’re having trouble with your order. Could you please provide the order ID so I can look up the details and help you resolve the issue?


KeyboardInterrupt: 